# Proyecto integrador · Fine-tuning de un Vision Transformer

Afinamos un **ViT** para clasificar imágenes y lo publicamos en el Hub. Une S36 (datasets), S38 (fine-tuning), S41 (visión) y S44 (publicar).

Objetivos:
1. Cargar y preprocesar un dataset de imágenes.
2. Afinar `ViTForImageClassification` con `Trainer`.
3. Evaluar y publicar en el Hub.

In [ ]:
%pip install -q transformers datasets evaluate accelerate

## 1 · Cargar y dividir el dataset

Usamos un subconjunto pequeño de `food101` (para ir rápido en CPU). Puedes cambiarlo por tu propio dataset (`imagefolder`).

In [ ]:
from datasets import load_dataset

ds = load_dataset("food101", split="train[:500]")
splits = ds.train_test_split(test_size=0.2)
train, test = splits["train"], splits["test"]

etiquetas = ds.features["label"].names
print("clases:", len(etiquetas), etiquetas[:5])

## 2 · Preprocesar con `ViTImageProcessor`

In [ ]:
from transformers import ViTImageProcessor

processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")

def transformar(ejemplos):
    inputs = processor([img.convert("RGB") for img in ejemplos["image"]], return_tensors="pt")
    inputs["label"] = ejemplos["label"]
    return inputs

train = train.with_transform(transformar)
test = test.with_transform(transformar)

## 3 · Modelo con cabeza de clasificación

In [ ]:
from transformers import ViTForImageClassification

modelo = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=len(etiquetas),
    ignore_mismatched_sizes=True,
)

## 4 · Entrenar con `Trainer`

In [ ]:
import numpy as np, evaluate
from transformers import TrainingArguments, Trainer, DefaultDataCollator

args = TrainingArguments(
    output_dir="./vit-alimentos",
    per_device_train_batch_size=16,
    evaluation_strategy="epoch",
    num_train_epochs=3,
    remove_unused_columns=False,
)

accuracy = evaluate.load("accuracy")

def metricas(eval_pred):
    logits, etiquetas = eval_pred
    return accuracy.compute(predictions=np.argmax(logits, -1), references=etiquetas)

trainer = Trainer(
    model=modelo, args=args,
    train_dataset=train, eval_dataset=test,
    data_collator=DefaultDataCollator(),
    compute_metrics=metricas,
)

trainer.train()
print(trainer.evaluate())

## 5 · Publicar en el Hub

Requiere token de escritura: `huggingface-cli login` (o `from huggingface_hub import login; login()`).

In [ ]:
# from huggingface_hub import login
# login()  # pega tu token de escritura

# trainer.save_model("./vit-alimentos")
# modelo.push_to_hub("tu-usuario/vit-alimentos")
# processor.push_to_hub("tu-usuario/vit-alimentos")

## Entregable

1. Notebook ejecutado (entrena y evalúa).
2. Modelo publicado en el Hub con su *model card*.

> Ampliación: despliega el modelo en un **Space con Gradio** (ver S44).